# Assignment problem

A fundamental combinatorial optimization problem.

- $n$ tasks to be done
- $n$ workers to do the tasks
- Each worker has a certain cost for each task ($c_{ij}$)

**Problem:** Find the best assignment of all tasks that minimizes the total cost.

**Constraints:**

- Each worker can do at most one task
- All tasks need to be done

# Mathematical Formulation

$$
\begin{aligned}
\text{minimize} \quad & \sum_{i=1}^{n} \sum_{j=1}^{n} c_{ij} x_{ij} \\
\text{subject to} \quad 
& \sum_{i=1}^{n} x_{ij} \le 1 
&& \forall j = 1, \ldots, n \quad \text{(workload)} \\
& \sum_{j=1}^{n} x_{ij} = 1 
&& \forall i = 1, \ldots, n \quad \text{(task completion)} \\
& x_{ij} \in \{0,1\} 
&& \forall i,j = 1, \ldots, n
\end{aligned}
$$

where $x_{ij}$ is $1$ if worker $j$ performs task $i$, and $0$ otherwise.

We can replace the integrality constraints with continuous constraints. The solution will not change as the constraint matrix is totally unimodular.

# Coding in Python using docplex

### Step 1: creating input data (const matrix)

In [1]:
import numpy as np

### Step 2: Importing docplex package

In [2]:
from docplex.mp.model import Model
cost = np.random.randint(1, 10, (4,4))

### Step 3: Creating the model

In [3]:
assignment_model = Model(name='Assignment')

### Step 4: Creating decision variables

In [4]:
x = assignment_model.binary_var_matrix(cost.shape[0], cost.shape[1], name='x')

### Step 5: Adding the constraints

In [9]:
# sum (i=1)^n x[i][j] <= 1 for all j
assignment_model.add_constraints((sum(x[i,j] for i in range(cost.shape[0])) <= 1 
                                  for j in range(cost.shape[1])),
                                 names = 'work_load')

# sum (j=1)^n x[i][j] = 1 for all i
assignment_model.add_constraints((sum(x[i,j] for j in range(cost.shape[1])) == 1 
                                  for i in range(cost.shape[0])),
                                 names = 'task_completion')

[docplex.mp.LinearConstraint[task_completion1](x_0_0+x_0_1+x_0_2+x_0_3,EQ,1),
 docplex.mp.LinearConstraint[task_completion2](x_1_0+x_1_1+x_1_2+x_1_3,EQ,1),
 docplex.mp.LinearConstraint[task_completion3](x_2_0+x_2_1+x_2_2+x_2_3,EQ,1),
 docplex.mp.LinearConstraint[task_completion4](x_3_0+x_3_1+x_3_2+x_3_3,EQ,1)]

### Step 6: Defining the objective function

In [11]:
obj_fn = sum(cost[i,j] * x[i,j] for i in range(cost.shape[0]) for j in range(cost.shape[1]))

assignment_model.set_objective('min', obj_fn)

assignment_model.print_information()

Model: Assignment
 - number of variables: 16
   - binary=16, integer=0, continuous=0
 - number of constraints: 8
   - linear=8
 - parameters: defaults
 - objective: minimize
 - problem type is: MILP


### Inspect model by printing

In [12]:
print(assignment_model.export_as_lp_string())

\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: Assignment

Minimize
 obj: 2 x_0_0 + 9 x_0_1 + 7 x_0_2 + 8 x_0_3 + 7 x_1_0 + x_1_1 + 7 x_1_2
      + 5 x_1_3 + 6 x_2_0 + 6 x_2_1 + 6 x_2_2 + 6 x_2_3 + x_3_0 + 5 x_3_1
      + 2 x_3_2 + 6 x_3_3
Subject To
 work_load1: x_0_0 + x_1_0 + x_2_0 + x_3_0 <= 1
 work_load2: x_0_1 + x_1_1 + x_2_1 + x_3_1 <= 1
 work_load3: x_0_2 + x_1_2 + x_2_2 + x_3_2 <= 1
 work_load4: x_0_3 + x_1_3 + x_2_3 + x_3_3 <= 1
 task_completion1: x_0_0 + x_0_1 + x_0_2 + x_0_3 = 1
 task_completion2: x_1_0 + x_1_1 + x_1_2 + x_1_3 = 1
 task_completion3: x_2_0 + x_2_1 + x_2_2 + x_2_3 = 1
 task_completion4: x_3_0 + x_3_1 + x_3_2 + x_3_3 = 1

Bounds
 0 <= x_0_0 <= 1
 0 <= x_0_1 <= 1
 0 <= x_0_2 <= 1
 0 <= x_0_3 <= 1
 0 <= x_1_0 <= 1
 0 <= x_1_1 <= 1
 0 <= x_1_2 <= 1
 0 <= x_1_3 <= 1
 0 <= x_2_0 <= 1
 0 <= x_2_1 <= 1
 0 <= x_2_2 <= 1
 0 <= x_2_3 <= 1
 0 <= x_3_0 <= 1
 0 <= x_3_1 <= 1
 0 <= x_3_2 <= 1
 0 <= x_3_3 <= 1

Binaries
 x_0_0 x_0_1 x_0_2 x_0_

### Step 7: Solve the model and output the solution

In [13]:
assignment_model.solve()

print('optimization is done. Objective function value: {}'.format(assignment_model.objective_value))

# get values of decision variables
assignment_model.print_solution()

optimization is done. Objective function value: 11.0
objective: 11
status: OPTIMAL_SOLUTION(2)
  x_0_0=1
  x_1_1=1
  x_2_3=1
  x_3_2=1


### Relaxing the binary constraints

In [15]:
assignment_model_2 = Model(name='assignment_2')

y = assignment_model_2.continuous_var_matrix(cost.shape[0], cost.shape[1], name='x')

assignment_model_2.add_constraints((sum(y[i,j] for i in range(cost.shape[0])) <= 1
                                    for j in range(cost.shape[1])),
                                   names = 'work_load')
assignment_model_2.add_constraints((sum(y[i,j] for j in range(cost.shape[1])) == 1
                                    for i in range(cost.shape[0])),
                                   names = 'task_completion')

obj_fn = sum(cost[i,j] * y[i,j] for i in range(cost.shape[0]) for j in range(cost.shape[1]))
assignment_model_2.set_objective('min', obj_fn)

assignment_model_2.solve()

print('optimization is done. Objective function value: {}'.format(assignment_model_2.objective_value))

assignment_model_2.print_solution()

optimization is done. Objective function value: 11.0
objective: 11.000
status: OPTIMAL_SOLUTION(2)
  x_0_0=1.000
  x_1_1=1.000
  x_2_3=1.000
  x_3_2=1.000


## LP relaxation

In mathematics, the **relaxation** of a **(mixed) integer linear program** is the problem that arises by removing the integrality constraint of each variable.

For example, in a **0–1 integer program**, all constraints are of the form

$$
x_i \in \{0,1\}.
$$

The relaxation of the original integer program instead uses a collection of linear constraints

$$
0 \le x_i \le 1.
$$

The resulting relaxation is a **linear program**, hence the name. This **relaxation technique** transforms an **NP-hard optimization problem** (integer programming) into a related problem that is solvable in **polynomial time** (linear programming); the solution to the relaxed linear program can be used to gain information about the solution to the original integer program.

The linear programming relaxation of an integer program may be solved using any standard linear programming technique. If it happens that, in the optimal solution, all variables have integer values, then it will also be an optimal solution to the original integer program. However, this is generally not true, except for some special cases (e.g. problems with totally unimodular matrix specifications.)



### Totally unimodular matrix specifications

Because **total unimodularity guarantees that every vertex/extreme point of the LP feasible region is integral**, as long as the right-hand side vector is integral.

Suppose the LP relaxation has the form

$$
\min c^T x
$$

subject to

$$
Ax \le b, \quad x \ge 0,
$$

where:

* $A$ is **totally unimodular**,
* $b$ is an **integer vector**.

A matrix $A$ is totally unimodular if every square submatrix has determinant $0$, $1$, or $-1$.

Now, an LP optimum occurs at an **extreme point** of the feasible polyhedron. At an extreme point, some constraints are active, so the solution is determined by a linear system

$$
B x_B = b_B,
$$

where $B$ is a square basis matrix formed from columns of the constraint matrix.

Since $A$ is totally unimodular, every nonsingular basis matrix $B$ has

$$
\det(B) = \pm 1.
$$

Using Cramer’s rule,

$$
x_B = B^{-1} b_B.
$$

Because $B^{-1}$ has integer entries when $\det(B)=\pm 1$, and because $b_B$ is integer, the basic solution $x_B$ must also be integer.

Therefore, every extreme point of the LP relaxation is integral. Since linear programs achieve their optima at extreme points, the LP relaxation has an optimal solution with integer-valued variables.

So the key logic is:

$$
\text{totally unimodular } A + \text{ integer } b
\Rightarrow \text{ integral LP vertices}
\Rightarrow \text{ LP relaxation has integer optimal solution.}
$$

For the assignment problem, this means replacing

$$
x_{ij} \in {0,1}
$$

with

$$
0 \le x_{ij} \le 1
$$

does not change the optimal solution: the LP relaxation will still return a solution where each $x_{ij}$ is $0$ or $1$.
